# Dexwin Pay — live assessment notebook

Read `README.md` and `data/DATA_CARD.md` first.

Goal: **catch fraud, cut false declines.** The cells below are a weak baseline. Improve them, then be ready to talk serving and monitoring.

Ignore `INTERVIEWER.md`.

In [1]:
!pip install numpy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_PATH = next(
    p for p in (Path("data/transactions.csv"), Path("../data/transactions.csv"))
    if p.exists()
)
df_raw = pd.read_csv(DATA_PATH)
df_raw.head(5)

,txn_id,event_ts,amount,currency,merchant_id,merchant_category,channel,country,customer_id,customer_tenure_days,txns_30d,avg_amount_30d,prior_declines_30d,hour,is_new_device,payment_method,device_fingerprint,ops_review_code,rules_engine_decline,is_fraud
0,TXN-000790,19/08/2026 22:38,9.42,NGN,M-2017,NaN,ios,NGA,C-10165,548.0,11,9.50,0,1.0,1,momo,DEV-45366,CNF,Y,1
1,TXN-002393,2026-08-09T11:21:00,32.23,GHS,M-2048,restaurants,ios,gh,C-10578,1.0,11,18.70,0,11.0,0,momo,DEV-27711,CLR,N,0
2,TXN-000145,03/09/2026 23:54,34.02,USD,M-2073,electronics,android,Kenya,C-10623,1442.0,8,49.68,1,1.0,0,card,DEV-14008,CLR,N,0
3,TXN-001134,05/09/2026 05:33,67.43,GHS,M-2041,NaN,android,ng,C-10283,720.0,34,88.69,0,12.0,0,card,DEV-83974,CLR,N,0
4,TXN-000464,17/08/2026 14:38,8.67,GHS,M-2016,RESTAURANTS,web,Ghana,C-10564,448.0,0,12.06,5,11.0,0,card,DEV-59655,CLR,Y,0


In [3]:
df_raw.shape

(3218, 20)

In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3218 entries, 0 to 3217
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   txn_id                3218 non-null   object 
 1   event_ts              3218 non-null   object 
 2   amount                2926 non-null   object 
 3   currency              3218 non-null   object 
 4   merchant_id           3218 non-null   object 
 5   merchant_category     2910 non-null   object 
 6   channel               3218 non-null   object 
 7   country               3218 non-null   object 
 8   customer_id           3218 non-null   object 
 9   customer_tenure_days  2993 non-null   float64
 10  txns_30d              3218 non-null   int64  
 11  avg_amount_30d        3218 non-null   float64
 12  prior_declines_30d    3218 non-null   int64  
 13  hour                  3081 non-null   float64
 14  is_new_device         3218 non-null   int64  
 15  payment_method       

In [ ]:
df_raw = df_raw.select_dtypes(include=['object'].columns)
for col in df_raw:
    df_raw[col]=df_raw[col].str.lower()

AttributeError: 'list' object has no attribute 'columns'

In [5]:
df_raw.describe()

,customer_tenure_days,txns_30d,avg_amount_30d,prior_declines_30d,hour,is_new_device,is_fraud
count,2993.000000,3218.000000,3218.000000,3218.000000,3081.000000,3218.000000,3218.000000
mean,951.343802,19.145432,67.746119,1.014916,11.537163,0.179925,0.084525
std,4089.720112,11.557535,294.699199,1.998234,6.829510,0.384185,0.278216
min,-3.000000,0.000000,1.840000,0.000000,0.000000,0.000000,0.000000
25%,282.000000,9.000000,15.670000,0.000000,6.000000,0.000000,0.000000
50%,743.000000,19.000000,29.455000,0.000000,12.000000,0.000000,0.000000
75%,1286.000000,29.000000,54.740000,1.000000,17.000000,0.000000,0.000000
max,99999.000000,39.000000,6869.320000,7.000000,23.000000,1.000000,1.000000


In [7]:
df_raw.isnull().sum()

txn_id                    0
event_ts                  0
amount                  292
currency                  0
merchant_id               0
merchant_category       308
channel                   0
country                   0
customer_id               0
customer_tenure_days    225
txns_30d                  0
avg_amount_30d            0
prior_declines_30d        0
hour                    137
is_new_device             0
payment_method            0
device_fingerprint        0
ops_review_code           0
rules_engine_decline      0
is_fraud                  0
dtype: int64

In [9]:
df_raw.duplicated().sum()

np.int64(18)

In [8]:
df_raw["merchant_category"].unique

<bound method Series.unique of 0               NaN
1       restaurants
2       electronics
3               NaN
4       RESTAURANTS
           ...     
3213    electronics
3214        telecom
3215         travel
3216        grocery
3217    restaurants
Name: merchant_category, Length: 3218, dtype: object>

In [16]:
df_raw["merchant_category"].isnull().mean

<bound method Series.mean of 0        True
1       False
2       False
3        True
4       False
        ...  
3213    False
3214    False
3215    False
3216    False
3217    False
Name: merchant_category, Length: 3218, dtype: bool>

In [ ]:
print(df_raw.dtypes)
print()
print(df_raw["is_fraud"].value_counts(normalize=True))

In [20]:
df_raw.drop_duplicates()

,txn_id,event_ts,amount,currency,merchant_id,merchant_category,channel,country,customer_id,customer_tenure_days,txns_30d,avg_amount_30d,prior_declines_30d,hour,is_new_device,payment_method,device_fingerprint,ops_review_code,rules_engine_decline,is_fraud
0,TXN-000790,19/08/2026 22:38,9.42,NGN,M-2017,NaN,ios,NGA,C-10165,548.0,11,9.50,0,1.0,1,momo,DEV-45366,CNF,Y,1
1,TXN-002393,2026-08-09T11:21:00,32.23,GHS,M-2048,restaurants,ios,gh,C-10578,1.0,11,18.70,0,11.0,0,momo,DEV-27711,CLR,N,0
2,TXN-000145,03/09/2026 23:54,34.02,USD,M-2073,electronics,android,Kenya,C-10623,1442.0,8,49.68,1,1.0,0,card,DEV-14008,CLR,N,0
3,TXN-001134,05/09/2026 05:33,67.43,GHS,M-2041,NaN,android,ng,C-10283,720.0,34,88.69,0,12.0,0,card,DEV-83974,CLR,N,0
4,TXN-000464,17/08/2026 14:38,8.67,GHS,M-2016,RESTAURANTS,web,Ghana,C-10564,448.0,0,12.06,5,11.0,0,card,DEV-59655,CLR,Y,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3213,TXN-001096,2026-08-14T09:09:00,30.36,GHS,M-2075,electronics,web,GH,C-10393,916.0,21,19.58,3,10.0,0,momo,DEV-53058,CLR,Y,0
3214,TXN-001131,21/07/2026 10:34,15.85,GHS,M-2054,telecom,ussd,NGA,C-10329,111.0,26,23.02,2,18.0,0,momo,DEV-60785,CLR,Y,0
3215,TXN-001295,2026-07-20T07:17:00,NaN,NGN,M-2046,travel,web,NG,C-10512,1140.0,38,1079.92,7,19.0,0,card,DEV-40256,CLR,Y,0
3216,TXN-000861,11/08/2026 01:35,155.61,GHS,M-2085,grocery,web,Ghana,C-10429,1202.0,15,85.74,0,14.0,0,card,DEV-88801,CLR,N,0


## Weak baseline

This matches `scripts/baseline.py`. It is a starting point, not a target design.

In [51]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
df = df_raw.copy()
n_loaded = len(df)

df["amount"] = (
    df["amount"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .replace("", pd.NA)
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df = df.dropna().copy()

y = df["is_fraud"]
X = df.drop(columns=["is_fraud"])

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=250)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"rows loaded:       {n_loaded}")
print(f"rows after dropna: {len(df)}")
print(f"accuracy:          {accuracy_score(y_test, pred):.4f}")
print(f"precision:          {precision_score(y_test, pred):.4f}")
print(f"recall:          {recall_score(y_test, pred):.4f}")
print(f"f1_score:          {f1_score(y_test, pred):.4f}")
print(confusion_matrix(y_test, pred))
# TODO: product asked for fewer false declines — add that later.

KeyError: 'amount'

## Your work

Clean the table, choose features that would exist at authorization time, pick a metric that matches the product, and train something you would consider shadowing.

Do not spend the whole session on hyperparameter search.

In [22]:
# Your cleaning /from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_PATH = next(
    p for p in (Path("data/transactions.csv"), Path("../data/transactions.csv"))
    if p.exists()
)
df_raw = pd.read_csv(DATA_PATH)
df_raw.head(5) 


,txn_id,event_ts,amount,currency,merchant_id,merchant_category,channel,country,customer_id,customer_tenure_days,txns_30d,avg_amount_30d,prior_declines_30d,hour,is_new_device,payment_method,device_fingerprint,ops_review_code,rules_engine_decline,is_fraud
0,TXN-000790,19/08/2026 22:38,9.42,NGN,M-2017,NaN,ios,NGA,C-10165,548.0,11,9.50,0,1.0,1,momo,DEV-45366,CNF,Y,1
1,TXN-002393,2026-08-09T11:21:00,32.23,GHS,M-2048,restaurants,ios,gh,C-10578,1.0,11,18.70,0,11.0,0,momo,DEV-27711,CLR,N,0
2,TXN-000145,03/09/2026 23:54,34.02,USD,M-2073,electronics,android,Kenya,C-10623,1442.0,8,49.68,1,1.0,0,card,DEV-14008,CLR,N,0
3,TXN-001134,05/09/2026 05:33,67.43,GHS,M-2041,NaN,android,ng,C-10283,720.0,34,88.69,0,12.0,0,card,DEV-83974,CLR,N,0
4,TXN-000464,17/08/2026 14:38,8.67,GHS,M-2016,RESTAURANTS,web,Ghana,C-10564,448.0,0,12.06,5,11.0,0,card,DEV-59655,CLR,Y,0


In [23]:
df_raw.shape

(3218, 20)

In [24]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3218 entries, 0 to 3217
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   txn_id                3218 non-null   object 
 1   event_ts              3218 non-null   object 
 2   amount                2926 non-null   object 
 3   currency              3218 non-null   object 
 4   merchant_id           3218 non-null   object 
 5   merchant_category     2910 non-null   object 
 6   channel               3218 non-null   object 
 7   country               3218 non-null   object 
 8   customer_id           3218 non-null   object 
 9   customer_tenure_days  2993 non-null   float64
 10  txns_30d              3218 non-null   int64  
 11  avg_amount_30d        3218 non-null   float64
 12  prior_declines_30d    3218 non-null   int64  
 13  hour                  3081 non-null   float64
 14  is_new_device         3218 non-null   int64  
 15  payment_method       

In [25]:
df_raw.describe()

,customer_tenure_days,txns_30d,avg_amount_30d,prior_declines_30d,hour,is_new_device,is_fraud
count,2993.000000,3218.000000,3218.000000,3218.000000,3081.000000,3218.000000,3218.000000
mean,951.343802,19.145432,67.746119,1.014916,11.537163,0.179925,0.084525
std,4089.720112,11.557535,294.699199,1.998234,6.829510,0.384185,0.278216
min,-3.000000,0.000000,1.840000,0.000000,0.000000,0.000000,0.000000
25%,282.000000,9.000000,15.670000,0.000000,6.000000,0.000000,0.000000
50%,743.000000,19.000000,29.455000,0.000000,12.000000,0.000000,0.000000
75%,1286.000000,29.000000,54.740000,1.000000,17.000000,0.000000,0.000000
max,99999.000000,39.000000,6869.320000,7.000000,23.000000,1.000000,1.000000


In [28]:
df_raw.duplicated().sum()

np.int64(18)

In [29]:
df_raw.drop_duplicates

<bound method DataFrame.drop_duplicates of           txn_id             event_ts     amount currency merchant_id  \
0     TXN-000790     19/08/2026 22:38       9.42      NGN      M-2017   
1     TXN-002393  2026-08-09T11:21:00      32.23      GHS      M-2048   
2     TXN-000145     03/09/2026 23:54      34.02      USD      M-2073   
3     TXN-001134     05/09/2026 05:33      67.43      GHS      M-2041   
4     TXN-000464     17/08/2026 14:38       8.67      GHS      M-2016   
...          ...                  ...        ...      ...         ...   
3213  TXN-001096  2026-08-14T09:09:00      30.36      GHS      M-2075   
3214  TXN-001131     21/07/2026 10:34      15.85      GHS      M-2054   
3215  TXN-001295  2026-07-20T07:17:00        NaN      NGN      M-2046   
3216  TXN-000861     11/08/2026 01:35     155.61      GHS      M-2085   
3217  TXN-003175     11/09/2026 05:20  82.12 GHS      GHS      M-2065   

     merchant_category  channel country customer_id  customer_tenure_days  \
0  

In [30]:
df_raw.isnull().sum()

txn_id                    0
event_ts                  0
amount                  292
currency                  0
merchant_id               0
merchant_category       308
channel                   0
country                   0
customer_id               0
customer_tenure_days    225
txns_30d                  0
avg_amount_30d            0
prior_declines_30d        0
hour                    137
is_new_device             0
payment_method            0
device_fingerprint        0
ops_review_code           0
rules_engine_decline      0
is_fraud                  0
dtype: int64

In [ ]:
df_raw=df['customer_tenure_days'] = df['customer_tenure_days'].fillna(df['customer_tenure_days'].mean())   

In [45]:
df_raw=df['hour'] = df['hour'].fillna(df['hour'].mode())   

In [46]:
df_raw=df['merchant_category'] = df['merchant_category'].fillna(df['merchant_category'].mode())   

In [47]:
df_raw=df['amount'] = df['amount'].fillna(df['amount'].mean())   


In [49]:
df_raw.isnull().sum

<bound method Series.sum of 1       False
2       False
4       False
5       False
6       False
        ...  
3212    False
3213    False
3214    False
3216    False
3217    False
Name: amount, Length: 2368, dtype: bool>

In [52]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
df = df_raw.copy()
n_loaded = len(df)

df["amount"] = (
    df["amount"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .replace("", pd.NA)
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df = df.dropna().copy()

y = df["is_fraud"]
X = df.drop(columns=["is_fraud"])

for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=250)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"rows loaded:       {n_loaded}")
print(f"rows after dropna: {len(df)}")
print(f"accuracy:          {accuracy_score(y_test, pred):.4f}")
print(f"precision:          {precision_score(y_test, pred):.4f}")
print(f"recall:          {recall_score(y_test, pred):.4f}")
print(f"f1_score:          {f1_score(y_test, pred):.4f}")
print(confusion_matrix(y_test, pred))
# TODO: product asked for fewer false declines — add that later.

KeyError: 'amount'

In [54]:
import joblib
joblib.dump(model, "fraud_detection.pkl")

['fraud_detection.pkl']

## Serving and monitoring (write bullets, then talk)

Freeze the model around here and answer in words:

1. What calls the model, and what is in the request?
2. Which features are **not** safe to serve online from this warehouse dump?
3. How do you roll it out without instantly changing every merchant’s decline rate?
4. Fraud labels arrive late. What do you monitor **tomorrow morning** vs next week?
5. How do you notice leakage / feedback loops once the model is live?

In [ ]:
1. After hosting model on platform you call it through an api
2. c


SyntaxError: invalid syntax (3194830181.py, line 1)